In [1]:
import pandas as pd
import looker_sdk
from io import StringIO

In [2]:
# Load the Looker Python SDK tools.
import looker_sdk

# Path to the `looker.ini` file we setup.


CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

# Retrieve my user information from Looker.
my_user = sdk.me()

# Display my username.
my_user["display_name"]

'Josh Allen'

In [3]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "clients.full_name",
        "clients.unique_identifier",
        "clients.id",
        "enrollments.ref_household",
        "household_makeup.household_type",
        "entry_screen.relationship_to_hoh",
        "clients.birth_date",
        "entry_screen.age",
        "clients.age",
        "agencies.name",
        "programs.name",
        "programs.project_type_code",
        "enrollments.id",
        "enrollments.start_date",
        "enrollments.end_date",
        "enrollments.days_since_start",
        "household_move_in_date.move_in_date",
        "members.assigned_staff",
        "static_demographics.veteran_text",
        "services.service_added_date",
        "client_notes_enrollment_level.note",
        "client_notes.id",
        "client_notes.added_date",
        "screens.name",
        "entry_screen.head_of_household",
        "enrollments.still_in_program",
        "client_assessments.assessment_date",
        "enrollments.added_date",
        "statusannual_assessment_count",
        "services.count",
        "client_notes_enrollment_level.count",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarters",
        "programs.status": "Active",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_1 = pd.read_csv(StringIO(result))

C:\Users\joallen\AppData\Local\Temp\ipykernel_30468\2372540842.py:62: DtypeWarning: Columns (14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df_1 = pd.read_csv(StringIO(result))


In [4]:
body = {
    "model": "oahu_connection_model",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "referrals.reassigned_date_date",
        "referrals.Currently_On_Queue",
        "referrals_referral_community_queues.name",
        "client_model.deleted",
        "enrollments.deleted",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarters",
        "programs.status": "Active",
        "client_model.deleted": "No",
        "enrollments.deleted": "No",
    },
    "limit": 100000,
}

result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_2 = pd.read_csv(StringIO(result))

In [5]:
rename_map = {"Unique Identifier": "Clients Unique Identifier"}

In [6]:
df_2 = df_2.rename(columns=rename_map)

In [7]:
df_3 = pd.merge(
    df_1,
    df_2,
    "left",
    on="Clients Unique Identifier",
)

In [8]:
df_2[df_2["Clients Unique Identifier"].duplicated()]

,Clients Unique Identifier,Referrals Reassigned Date,Referrals Is Referral Currently On Queue?,Referrals Community Queue Name,Clients Deleted (Yes / No),Enrollments Deleted (Yes / No)
14,06CB14A1D,2024-08-01,No,Oahu Housing Queue,No,No
68,7A2527A5B,2024-07-05,No,Oahu Housing Queue,No,No
96,A47B74C3A,2024-06-25,No,Oahu Housing Queue,No,No
97,01CC3F145,2024-06-25,No,Oahu Housing Queue,No,No
106,B4760B29A,2024-06-21,No,Oahu Housing Queue,No,No
...,...,...,...,...,...,...
9713,A47B74C3A,NaN,No,PIC EHV Queue,No,No
9719,649221F95,NaN,Yes,Oahu Housing Queue,No,No
9720,0071595D4,NaN,Yes,Oahu Housing Queue,No,No
9726,1E32AC82C,NaN,No,Oahu Housing Queue,No,No


In [9]:
df_1[df_1["Clients Unique Identifier"].duplicated()]

,Clients Client Full Name,Clients Unique Identifier,Clients Client ID,Enrollments Household ID,Enrollments Household Type,Entry Screen Relationship to Head of Household,Clients Date of Birth Date,Entry Screen Age at Project Start,Clients Current Age,Agencies Agency Name,...,Client Notes - Enrollment Level Note,Client Notes - Client Level ID,Client Notes - Client Level Date Added Date,Client Assessments Name,Entry Screen Head of Household (Yes / No),Enrollments Active in Project,Client Assessments Assessment Date,Enrollments Date Created Date,Services Count,Client Notes - Enrollment Level Count
2,Jaynor Neor,C8ED6DE28,51451,141839,Household with Children,Head of household’s child,2024-07-09,0,0,U.S. Veterans Initiative,...,NaN,NaN,NaN,NaN,No,Yes,NaN,2024-08-08,0,0
4,Robby-Gordon Miles Maiava-Faasalaiai,24E512A80,51316,138164,Household with Children,Head of household’s child,2024-07-09,0,0,Alternative Structures International,...,NaN,NaN,NaN,NaN,No,Yes,NaN,2024-07-24,2,0
5,Robby-Gordon Miles Maiava-Faasalaiai,24E512A80,51316,141448,Single Child,Head of household’s child,2024-07-09,0,0,Alternative Structures International,...,NaN,NaN,NaN,NaN,No,No,NaN,2024-07-24,0,0
8,Zariah Sausi,B9CE9338A,51020,140536,Household with Children,Head of household’s child,2024-06-22,0,0,Alternative Structures International,...,NaN,NaN,NaN,NaN,No,Yes,NaN,2024-06-26,1,0
9,Zariah Sausi,B9CE9338A,51020,140536,Household with Children,Head of household’s child,2024-06-22,0,0,Alternative Structures International,...,NaN,NaN,NaN,NaN,No,Yes,NaN,2024-06-26,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Lina Thongsonlone,2F737E5E6,5135,110858,Single Adult,Self (head of household),1980-11-18,41,43,Honolulu Community Action Program,...,Working on getting SSDI. Once obtained LT can ...,32197.0,2021-09-14,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2021-08-02,2022-04-08,3,1
99996,Lina Thongsonlone,2F737E5E6,5135,110858,Single Adult,Self (head of household),1980-11-18,41,43,Honolulu Community Action Program,...,Working on getting SSDI. Once obtained LT can ...,32197.0,2021-09-14,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2021-10-08,2022-04-08,3,1
99997,Lina Thongsonlone,2F737E5E6,5135,110858,Single Adult,Self (head of household),1980-11-18,41,43,Honolulu Community Action Program,...,Working on getting SSDI. Once obtained LT can ...,32197.0,2021-09-14,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2021-12-22,2022-04-08,3,1
99998,Lina Thongsonlone,2F737E5E6,5135,110858,Single Adult,Self (head of household),1980-11-18,41,43,Honolulu Community Action Program,...,Working on getting SSDI. Once obtained LT can ...,32197.0,2021-09-14,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2022-06-08,2022-04-08,3,1


In [57]:
df_3[df_3["Clients Unique Identifier"].duplicated()]

,Clients Client Full Name,Clients Unique Identifier,Clients Client ID,Enrollments Household ID,Enrollments Household Type,Entry Screen Relationship to Head of Household,Clients Date of Birth Date,Entry Screen Age at Project Start,Clients Current Age,Agencies Agency Name,...,Client Assessments Name,Entry Screen Head of Household (Yes / No),Enrollments Active in Project,Client Assessments Assessment Date,Enrollments Date Created Date,Services Count,Client Notes - Enrollment Level Count,Referrals Date Referral Was Reassigned Date,Referrals Is Referral Currently On Queue?,Referrals Community Queue Name
2,Robby-Gordon Miles Maiava-Faasalaiai,24E512A80,51316,138164,Household with Children,Head of household’s child,2024-07-09,0,0,Alternative Structures International,...,NaN,No,Yes,NaN,2024-07-24,2,0,NaN,NaN,NaN
3,Robby-Gordon Miles Maiava-Faasalaiai,24E512A80,51316,141448,Single Child,Head of household’s child,2024-07-09,0,0,Alternative Structures International,...,NaN,No,No,NaN,2024-07-24,0,0,NaN,NaN,NaN
6,Zariah Sausi,B9CE9338A,51020,140536,Household with Children,Head of household’s child,2024-06-22,0,0,Alternative Structures International,...,NaN,No,Yes,NaN,2024-06-26,1,0,NaN,NaN,NaN
7,Zariah Sausi,B9CE9338A,51020,140536,Household with Children,Head of household’s child,2024-06-22,0,0,Alternative Structures International,...,NaN,No,Yes,NaN,2024-06-26,3,0,NaN,NaN,NaN
8,Zariah Sausi,B9CE9338A,51020,140536,Household with Children,Head of household’s child,2024-06-22,0,0,Alternative Structures International,...,NaN,No,Yes,NaN,2024-06-26,1,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140788,Jontae Jones,EE39C21DA,34952,104828,Single Adult,Self (head of household),1980-07-28,41,44,Honolulu Community Action Program,...,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2022-10-26,2021-11-19,1,1,NaN,NaN,NaN
140789,Jontae Jones,EE39C21DA,34952,104828,Single Adult,Self (head of household),1980-07-28,41,44,Honolulu Community Action Program,...,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2021-10-25,2021-11-19,1,1,NaN,NaN,NaN
140790,Jontae Jones,EE39C21DA,34952,104828,Single Adult,Self (head of household),1980-07-28,41,44,Honolulu Community Action Program,...,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2022-10-26,2021-11-19,1,1,NaN,NaN,NaN
140791,Jontae Jones,EE39C21DA,34952,104828,Single Adult,Self (head of household),1980-07-28,41,44,Honolulu Community Action Program,...,Custom VI-SPDAT Individual Adult (V2),Yes,Yes,2021-10-25,2021-11-19,1,1,NaN,NaN,NaN


In [10]:
from ds_sql_connections import GsAutomationReliefAssistanceConnection
import os
import json

main_directory_path = str(os.getcwd())
# Path to the config file
config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
# creating the config object to pull the authentication credentials later in various calls.
with open(config_path, "r", encoding="utf-8") as file:
    config_object = json.load(file)

# sql connection credentials
sql_username = config_object["Josh"]["username"]
sql_password = config_object["Josh"]["password"]
conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)
conn.insert_with_progress(
    df=df_3,
    schema="dbo",
    tbl_nm="partners_in_care_active_projects",
    dtypes={},
)

In [13]:
df_1.to_excel("partners_in_care_active_projects.xlsx")

In [13]:
look = sdk.look("92496")

In [17]:
filter = look.query["filters"]

In [20]:
columns = look.query["fields"]

In [21]:
# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": columns,
    "filters": filter,
    "limit": 10000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df = pd.read_csv(StringIO(result))

In [22]:
df

,Clients Client ID,Clients Personal ID,Clients Unique Identifier,Clients Client Full Name,Clients Current Age
0,57,23e370b8c7f24958a239f5c862200948,3F4B81232,Donovan Vaimasanuu,21.0
1,63,322f27212d1b46c1aa66800f8d0240df,D90269041,Maxine Lopes,45.0
2,110,8607af49da58435d9d60b364feb8df6e,1748D5775,Cyrus Clark-Cordeiro,29.0
3,128,54a1a08ef57140109ac9b8f9e8fdae24,5969492AD,Viliamu Pulusila,67.0
4,133,7d9205dfa4a54259b95c82b7cc1d746e,9F7506363,Rosemarie Gacho,40.0
...,...,...,...,...,...
4060,51161,3df869fb647e4f939a4cc7b020338fa4,C6C96FE95,Susiann Sos,7.0
4061,51169,1341756e6c944069b3ee820314b6ed2c,D84504FD9,Michael Wallis,47.0
4062,51183,ef4a0b16b1d84a31ace7a5b91315a640,6D46B55D2,Wilsand Sardinha,35.0
4063,51190,d692f52384f24bc2bf191c0ab21cbc9f,D685E7597,Junior Richard,24.0


In [1]:
import json
import sys
from typing import cast, Dict, List, Union

import looker_sdk
from looker_sdk import models40 as models, error

In [2]:
def main() -> None:
    look_id = "92793"
    filter_args = iter(sys.argv[2:])
    filters: Dict[str, str] = {}

    if not (look_id and len(sys.argv[2:]) > 0 and len(sys.argv[2:]) % 2 == 0):
        raise Exception(
            "Please provide: <lookId> <filter_1> <filter_value_1> "
            "<filter_2> <filter_value_2> ..."
        )

    for filter_name in filter_args:
        filters[filter_name] = next(filter_args)

    query = get_look_query(int(look_id))
    results = run_query_with_filter(query, filters)

    print(f"Query results with filters={filters}:\n{results}", end="\n\n")


def get_look_query(id: int) -> models.Query:
    """Returns the query associated with a given look id."""
    try:
        look = sdk.look(id)
    except error.SDKError:
        raise Exception(f"Error getting Look {id}")
    else:
        query = look.query
    return query


TJson = List[Dict[str, Union[str, int, float, bool, None]]]


def run_query_with_filter(query: models.Query, filters: Dict[str, str]) -> TJson:
    """Runs the specified query with the specified filters."""
    request = create_query_request(query, filters)
    try:
        json_ = sdk.run_inline_query("json", request, cache=False)
    except error.SDKError:
        raise Exception("Error running query")
    else:
        json_resp = cast(TJson, json.loads(json_))
    return json_resp


def create_query_request(q: models.Query, filters: Dict[str, str]) -> models.WriteQuery:
    return models.WriteQuery(
        model=q.model,
        view=q.view,
        fields=q.fields,
        pivots=q.pivots,
        fill_fields=q.fill_fields,
        filters=filters,
        sorts=q.sorts,
        limit=q.limit,
        column_limit=q.column_limit,
        total=q.total,
        row_total=q.row_total,
        subtotals=q.subtotals,
        dynamic_fields=q.dynamic_fields,
        query_timezone=q.query_timezone,
    )

In [7]:
CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

FileNotFoundError: No config file found: 'utils\looker.ini'

In [6]:
sdk.look("92793")

NameError: name 'sdk' is not defined